### 0 SETUP

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import anthropic
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [2]:
from dotenv import load_dotenv
import os, anthropic

load_dotenv(override=True)   

key = os.getenv("ANTHROPIC_API_KEY")
print(repr(key[:10]), len(key), repr(key[-4:]))

client = anthropic.Anthropic()
print(repr(client.api_key[:10]), len(client.api_key))

'sk-ant-usr' 106 '9QAA'
'sk-ant-usr' 106


In [3]:
print(os.getenv("ANTHROPIC_API_KEY") is not None)

True


### Import Data

In [4]:
df = pd.read_csv("IMDB_Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
IMDB_Dataset_df = df.copy()

In [6]:
IMDB_Dataset_df.shape

(50000, 2)

In [7]:
IMDB_Dataset_df = IMDB_Dataset_df.sample(300, random_state=1).reset_index(drop=True)

In [8]:
IMDB_Dataset_df.shape

(300, 2)

In [9]:
IMDB_Dataset_df = IMDB_Dataset_df[['review', 'sentiment']]

In [10]:
IMDB_Dataset_df.head()

,review,sentiment
0,With No Dead Heroes you get stupid lines like ...,negative
1,I thought maybe... maybe this could be good. A...,negative
2,An elite American military team which of cours...,negative
3,Ridiculous horror film about a wealthy man (Jo...,negative
4,"Well, if you are one of those Katana's film-nu...",positive


### Claude - Haiku 5-5

In [11]:
system_prompt = """Classify the overall sentiment expressed in the movie review.
Reply with exactly one lowercase word: positive, neutral, or negative.

Use positive or negative when the review clearly expresses an overall opinion in that direction.
Use neutral when the opinion is genuinely mixed, unclear, or mostly factual.
Consider the review's meaning, including sarcasm; do not classify based only on isolated positive or negative words."""

In [12]:
LABELS = ['positive', 'neutral', 'negative']

In [13]:
def compute_claude_sentiment(text):
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=10,
        thinking={"type": "disabled"},
        system=system_prompt,
        messages=[{"role": "user", "content": f"Review: {text}"}]
    )
    answer = next(
        (block.text for block in response.content if block.type == "text"),
        ""
    ).strip().strip(".").lower()

    return answer if answer in LABELS else "neutral"

In [14]:
compute_claude_sentiment('I absolutely loved every minute of this movie')

'positive'

### Using Haiku

In [15]:
IMDB_Dataset_df['claude_sentiment'] = IMDB_Dataset_df['review'].apply(compute_claude_sentiment)

IMDB_Dataset_df[['review', 'sentiment', 'claude_sentiment']].head()

,review,sentiment,claude_sentiment
0,With No Dead Heroes you get stupid lines like ...,negative,negative
1,I thought maybe... maybe this could be good. A...,negative,negative
2,An elite American military team which of cours...,negative,negative
3,Ridiculous horror film about a wealthy man (Jo...,negative,negative
4,"Well, if you are one of those Katana's film-nu...",positive,positive


In [16]:
IMDB_Dataset_df

,review,sentiment,claude_sentiment
0,With No Dead Heroes you get stupid lines like ...,negative,negative
1,I thought maybe... maybe this could be good. A...,negative,negative
2,An elite American military team which of cours...,negative,negative
3,Ridiculous horror film about a wealthy man (Jo...,negative,negative
4,"Well, if you are one of those Katana's film-nu...",positive,positive
...,...,...,...
295,Guy Pearce looks like and acts like a Calvin K...,negative,negative
296,"In an otherwise good review, loleralacartelort...",negative,neutral
297,"While the original titillates the intellect, t...",negative,negative
298,"I am a Jane Eyre lover and a purist, and this ...",positive,positive


In [17]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

HUMAN_LABELS = ["negative", "positive"]

y_true = IMDB_Dataset_df["sentiment"]
y_pred = IMDB_Dataset_df["claude_sentiment"]

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=HUMAN_LABELS,
    average="macro",
)

print(f"Accuracy: {accuracy:.3f}")
print(f"Macro-F1: {macro_f1:.3f}")

print(
    classification_report(
        y_true,
        y_pred,
        labels=["negative", "neutral", "positive"],
        zero_division=0,
    )
)

Accuracy: 0.920
Macro-F1: 0.950
              precision    recall  f1-score   support

    negative       0.98      0.95      0.96       152
     neutral       0.00      0.00      0.00         0
    positive       0.99      0.89      0.94       148

    accuracy                           0.92       300
   macro avg       0.65      0.61      0.63       300
weighted avg       0.98      0.92      0.95       300



### Vader Sentiment (Comparison)

In [18]:
sia = SentimentIntensityAnalyzer()

def compute_vader_sentiment(text):
    score = sia.polarity_scores(text)['compound']

    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    return 'neutral'

In [19]:
IMDB_Dataset_df['vader_sentiment'] = IMDB_Dataset_df['review'].apply(compute_vader_sentiment)

In [20]:
IMDB_Dataset_df

,review,sentiment,claude_sentiment,vader_sentiment
0,With No Dead Heroes you get stupid lines like ...,negative,negative,negative
1,I thought maybe... maybe this could be good. A...,negative,negative,positive
2,An elite American military team which of cours...,negative,negative,positive
3,Ridiculous horror film about a wealthy man (Jo...,negative,negative,negative
4,"Well, if you are one of those Katana's film-nu...",positive,positive,positive
...,...,...,...,...
295,Guy Pearce looks like and acts like a Calvin K...,negative,negative,positive
296,"In an otherwise good review, loleralacartelort...",negative,neutral,negative
297,"While the original titillates the intellect, t...",negative,negative,positive
298,"I am a Jane Eyre lover and a purist, and this ...",positive,positive,positive


### Model Comparison

In [21]:
pd.crosstab(
    IMDB_Dataset_df['sentiment'],
    IMDB_Dataset_df['claude_sentiment']
)

claude_sentiment,negative,neutral,positive
sentiment,,,
negative,144,6,2
positive,3,13,132


In [22]:
pd.crosstab(
    IMDB_Dataset_df["sentiment"],
    IMDB_Dataset_df["vader_sentiment"]
)

vader_sentiment,negative,neutral,positive
sentiment,,,
negative,89,2,61
positive,20,0,128


In [23]:
IMDB_Dataset_df.to_csv('Sentiment_Analysis_IMDB_Dataset_with_claude.csv', index=False)